# Cryptarithm & Equation Puzzle Solver
Solves 1555 puzzles from train.csv using formula families from CRYPTARITHM_SOLVER_RULES.md

In [ ]:
import csv, re, itertools
from collections import defaultdict

In [ ]:
# ── Parse puzzles from train.csv ──────────────────────────────────────────────

CSV_PATH = "data/raw/train.csv"

def parse_puzzles(path):
    """Extract cryptarithm/equation puzzles from train.csv."""
    puzzles = []
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            if "secret set of transformation rules" not in row["prompt"]:
                continue
            lines = row["prompt"].strip().split("\n")
            examples = []
            query = None
            for line in lines:
                line = line.strip()
                if "determine the result for:" in line.lower():
                    query = line.split(":")[-1].strip()
                elif "=" in line and "transformation" not in line.lower() and len(line) < 80:
                    # Parse equation: LHS = RHS
                    parts = line.split("=", 1)
                    if len(parts) == 2:
                        lhs = parts[0].strip()
                        rhs = parts[1].strip()
                        if len(lhs) == 5:  # AB op CD format
                            examples.append({"lhs": lhs, "rhs": rhs, "raw": line})
            if query and len(query) == 5:
                puzzles.append({
                    "id": row["id"],
                    "answer": row["answer"],
                    "examples": examples,
                    "query": query,
                })
    return puzzles

puzzles = parse_puzzles(CSV_PATH)
print(f"Parsed {len(puzzles)} puzzles")

# Classify numeric vs symbolic
numeric_puzzles = []
symbolic_puzzles = []
for p in puzzles:
    q = p["query"]
    if q[:2].isdigit() and q[3:5].isdigit():
        numeric_puzzles.append(p)
    else:
        symbolic_puzzles.append(p)
print(f"Numeric: {len(numeric_puzzles)}, Symbolic: {len(symbolic_puzzles)}")

# Show sample
for p in puzzles[:2]:
    print(f"\nID={p['id']}, query={p['query']}, answer={p['answer']}")
    for e in p["examples"]:
        print(f"  {e['raw']}")

In [ ]:
# ── Formula Definitions (from CRYPTARITHM_SOLVER_RULES.md) ────────────────────
# All formulas return STRING results. rev() returns reversed string preserving
# leading zeros from reversal (rev(1000)="0001", rev(70)="07"). No zero-padding.

def rev_s(x):
    """Reverse string representation. rev(1000)='0001', rev(70)='07'."""
    return str(x)[::-1]

def rev2_i(x):
    """Reverse 2-digit representation as int. rev2(8)=80, rev2(70)=7."""
    return int(f'{int(x):02d}'[::-1])

def safe_mod(a, b):
    return a % b if b != 0 else None

FORMULAS = []
def F(fam, name, fn, sp=False):
    FORMULAS.append((fam, name, fn, sp))

# DIRECT_MAXMIN (57%)
F('DM','max+min',       lambda a,b: str(max(a,b)+min(a,b)))
F('DM','max*min',       lambda a,b: str(max(a,b)*min(a,b)))
F('DM','max-min',       lambda a,b: str(max(a,b)-min(a,b)))
F('DM','max||min',      lambda a,b: f'{max(a,b)}{min(a,b)}')
F('DM','min||max',      lambda a,b: f'{min(a,b)}{max(a,b)}')
F('DM','(max*min)+1',   lambda a,b: str(max(a,b)*min(a,b)+1))
F('DM','(max*min)-1',   lambda a,b: str(max(a,b)*min(a,b)-1))
F('DM','(max+min)+1',   lambda a,b: str(max(a,b)+min(a,b)+1))
F('DM','(max+min)-1',   lambda a,b: str(max(a,b)+min(a,b)-1))
F('DM','max%min',       lambda a,b: str(safe_mod(max(a,b),min(a,b))) if min(a,b)!=0 else None)
F('DM','(max-min)+1',   lambda a,b: str(max(a,b)-min(a,b)+1))
F('DM','(max-min)-1',   lambda a,b: str(max(a,b)-min(a,b)-1))
F('DM','sp:max-min',    lambda a,b: str(max(a,b)-min(a,b)), True)
F('DM','sp:(max-min-1)',lambda a,b: str(max(a,b)-min(a,b)-1), True)
F('DM','sp:(max-min+1)',lambda a,b: str(max(a,b)-min(a,b)+1), True)

# REV_AB (29%) — rev_s preserves leading zeros naturally
F('RA','rev(ra*rb)',    lambda a,b: rev_s(rev2_i(a)*rev2_i(b)))
F('RA','rev(ra-rb)',    lambda a,b: rev_s(abs(rev2_i(a)-rev2_i(b))))
F('RA','rev(ra+rb)',    lambda a,b: rev_s(rev2_i(a)+rev2_i(b)))
F('RA','rev(ra+rb+1)',  lambda a,b: rev_s(rev2_i(a)+rev2_i(b)+1))
F('RA','rev(ra||rb)',   lambda a,b: rev_s(int(f'{rev2_i(a)}{rev2_i(b)}')))
F('RA','rev(rb-ra)',    lambda a,b: rev_s(abs(rev2_i(b)-rev2_i(a))))
F('RA','rev(ra*rb+1)',  lambda a,b: rev_s(rev2_i(a)*rev2_i(b)+1))
F('RA','rev(ra*rb-1)',  lambda a,b: rev_s(rev2_i(a)*rev2_i(b)-1))
F('RA','rev(ra+rb-1)',  lambda a,b: rev_s(rev2_i(a)+rev2_i(b)-1))
F('RA','rev(rb||ra)',   lambda a,b: rev_s(int(f'{rev2_i(b)}{rev2_i(a)}')))
F('RA','sp:rev(ra-rb)', lambda a,b: rev_s(abs(rev2_i(a)-rev2_i(b))), True)

# DIRECT_SIMPLE (11%)
F('DS','a-b',     lambda a,b: str(a-b))
F('DS','a||b',    lambda a,b: f'{a}{b}')
F('DS','a*b',     lambda a,b: str(a*b))
F('DS','a+b',     lambda a,b: str(a+b))
F('DS','(a+b)+1', lambda a,b: str(a+b+1))
F('DS','(a+b)-1', lambda a,b: str(a+b-1))
F('DS','(a*b)+1', lambda a,b: str(a*b+1))
F('DS','(a*b)-1', lambda a,b: str(a*b-1))
F('DS','a%b',     lambda a,b: str(safe_mod(a,b)) if b!=0 else None)
F('DS','b%a',     lambda a,b: str(safe_mod(b,a)) if a!=0 else None)
F('DS','b||a',    lambda a,b: f'{b}{a}')
F('DS','b-a',     lambda a,b: str(b-a))
F('DS','sp:a-b',  lambda a,b: str(abs(a-b)), True)

# REV_MAXMIN (2%)
F('RM','rev(rmax*rmin)',  lambda a,b: rev_s(max(rev2_i(a),rev2_i(b))*min(rev2_i(a),rev2_i(b))))
F('RM','rev(rmax+rmin)',  lambda a,b: rev_s(max(rev2_i(a),rev2_i(b))+min(rev2_i(a),rev2_i(b))))
F('RM','rev(rmax-rmin)',  lambda a,b: rev_s(max(rev2_i(a),rev2_i(b))-min(rev2_i(a),rev2_i(b))))
F('RM','rev(rmax%rmin)',  lambda a,b: rev_s(safe_mod(max(rev2_i(a),rev2_i(b)),min(rev2_i(a),rev2_i(b)))) if min(rev2_i(a),rev2_i(b))!=0 else None)
F('RM','rev(rmax||rmin)', lambda a,b: rev_s(int(f'{max(rev2_i(a),rev2_i(b))}{min(rev2_i(a),rev2_i(b))}')))

def matches(val_str, result_str, op_char, sp):
    """Check if formula output matches displayed result. No padding tricks."""
    if val_str is None: return False
    if sp:
        return op_char + val_str == result_str
    if val_str == result_str: return True
    if result_str == op_char + val_str: return True  # sign-prefix display
    if result_str == val_str + op_char: return True   # sign-suffix display
    return False

print(f"Loaded {len(FORMULAS)} formula variants")

In [ ]:
# ── Solver ────────────────────────────────────────────────────────────────────

import time

def find_matching_formulas(eqs):
    """For equations sharing same operator, find all formulas matching ALL."""
    op_char = eqs[0]['lhs'][2]
    found = []
    for fam, name, fn, sp in FORMULAS:
        ok = True
        for eq in eqs:
            a, b = int(eq['lhs'][:2]), int(eq['lhs'][3:5])
            try:
                val = fn(a, b)
                if not matches(val, eq['rhs'], op_char, sp):
                    ok = False; break
            except: ok = False; break
        if ok: found.append((fam, name, fn, sp))
    return found

def solve_numeric(puzzle):
    """Solve puzzle with digit operands."""
    ops = defaultdict(list)
    for eq in puzzle['examples']:
        ops[eq['lhs'][2]].append(eq)
    
    query = puzzle['query']
    q_a, q_b = int(query[:2]), int(query[3:5])
    q_op = query[2]
    answer = puzzle['answer']
    
    op_matches = {oc: find_matching_formulas(eqs) for oc, eqs in ops.items()}
    valid_fams = None
    for oc, found in op_matches.items():
        fams = set(f[0] for f in found)
        valid_fams = fams if valid_fams is None else valid_fams & fams
    if not valid_fams: valid_fams = set()
    
    candidate_lists = []
    if q_op in op_matches:
        if valid_fams:
            candidate_lists.append([c for c in op_matches[q_op] if c[0] in valid_fams])
        candidate_lists.append(op_matches[q_op])
    if valid_fams:
        candidate_lists.append([f for f in FORMULAS if f[0] in valid_fams])
    candidate_lists.append(FORMULAS)
    
    for candidates in candidate_lists:
        for fam, name, fn, sp in candidates:
            try:
                val = fn(q_a, q_b)
                if matches(val, answer, q_op, sp):
                    return answer
            except: continue
    return ''


# ── Symbolic Solver (constraint-based, fast) ─────────────────────────────────

def extract_result_digits(rhs, op_char):
    """Parse result string into (body_chars, is_sign_prefix, is_sign_suffix).
    body_chars are the symbol characters that map to result digits.
    """
    if len(rhs) >= 2 and rhs[0] == op_char:
        return list(rhs[1:]), True, False
    if len(rhs) >= 2 and rhs[-1] == op_char:
        return list(rhs[:-1]), False, True
    return list(rhs), False, False

def result_to_digits(val_str):
    """Convert formula output string to list of digit ints."""
    return [int(c) for c in val_str]

def get_candidate_mappings(eq, op_chars):
    """For one equation, enumerate all (a,b) + formula combos.
    Returns list of (partial_mapping_dict, formula_index).
    partial_mapping maps symbol -> digit for all symbols in this equation.
    """
    lhs, rhs = eq['lhs'], eq['rhs']
    oc = lhs[2]
    a_syms = [lhs[0], lhs[1]]
    b_syms = [lhs[3], lhs[4]]
    rhs_body, is_sp, is_sf = extract_result_digits(rhs, oc)
    
    candidates = []
    
    for a in range(100):
        a_digits = [a // 10, a % 10]
        for b in range(100):
            b_digits = [b // 10, b % 10]
            
            # Build partial mapping from operand symbols
            partial = {}
            conflict = False
            for sym, d in zip(a_syms + b_syms, a_digits + b_digits):
                if sym in op_chars: continue  # skip operator chars in operand position
                if sym in partial:
                    if partial[sym] != d:
                        conflict = True; break
                else:
                    partial[sym] = d
            if conflict: continue
            
            # Check injectivity (different symbols -> different digits)
            if len(set(partial.values())) != len(partial):
                continue
            
            # Try each formula
            for fi, (fam, name, fn, sp) in enumerate(FORMULAS):
                try:
                    val_str = fn(a, b)
                except:
                    continue
                if val_str is None: continue
                
                # Check sign-prefix/suffix compatibility
                if sp and not is_sp: continue
                if not sp and is_sp:
                    # Formula doesn't flag sign-prefix but display has it — some formulas overlap
                    # Try matching anyway: val_str should match body
                    pass
                
                # Check result digits match rhs symbols
                r_digits = list(val_str)
                
                if sp:
                    if len(r_digits) != len(rhs_body): continue
                elif is_sp:
                    # Display has sign-prefix, formula doesn't flag it
                    # The absolute value should match
                    abs_str = str(abs(int(val_str))) if val_str.lstrip('-').isdigit() else val_str
                    r_digits = list(abs_str)
                    if len(r_digits) != len(rhs_body): continue
                elif is_sf:
                    if len(r_digits) != len(rhs_body): continue
                else:
                    if len(r_digits) != len(rhs_body): continue
                
                # Map result symbols to digits
                mapping = dict(partial)
                ok = True
                for sym, d_ch in zip(rhs_body, r_digits):
                    d = int(d_ch)
                    if sym in op_chars: continue
                    if sym in mapping:
                        if mapping[sym] != d:
                            ok = False; break
                    else:
                        mapping[sym] = d
                if not ok: continue
                
                # Check injectivity
                vals = list(mapping.values())
                if len(set(vals)) != len(vals): continue
                
                candidates.append((mapping, fi))
    
    return candidates

def solve_symbolic(puzzle):
    """Solve puzzle with symbol operands using constraint intersection."""
    examples = puzzle['examples']
    query = puzzle['query']
    q_op = query[2]
    answer = puzzle['answer']
    
    op_chars = set(eq['lhs'][2] for eq in examples) | {q_op}
    
    # Collect all symbols
    all_symbols = set()
    for eq in examples:
        for ch in eq['lhs'][0:2] + eq['lhs'][3:5]:
            if ch not in op_chars: all_symbols.add(ch)
        for ch in eq['rhs']:
            if ch not in op_chars: all_symbols.add(ch)
    for ch in query[0:2] + query[3:5] + answer:
        if ch not in op_chars: all_symbols.add(ch)
    
    n = len(all_symbols)
    if n > 10: return ''
    
    # Get candidates from first equation (most constrained)
    eq0 = examples[0]
    candidates = get_candidate_mappings(eq0, op_chars)
    
    if not candidates:
        return ''
    
    # Filter candidates against remaining equations
    for eq in examples[1:]:
        lhs, rhs = eq['lhs'], eq['rhs']
        oc = lhs[2]
        a_syms = [lhs[0], lhs[1]]
        b_syms = [lhs[3], lhs[4]]
        rhs_body, is_sp, is_sf = extract_result_digits(rhs, oc)
        
        new_candidates = []
        for mapping, fi0 in candidates:
            # Decode a, b from mapping
            a_digits = []
            b_digits = []
            ok = True
            for sym in a_syms:
                if sym in op_chars:
                    ok = False; break
                if sym in mapping:
                    a_digits.append(mapping[sym])
                else:
                    ok = False; break
            if not ok:
                # Symbol not yet mapped — need to try all values
                # For now, keep candidate and check later
                new_candidates.append((mapping, fi0))
                continue
            for sym in b_syms:
                if sym in op_chars:
                    ok = False; break
                if sym in mapping:
                    b_digits.append(mapping[sym])
                else:
                    ok = False; break
            if not ok:
                new_candidates.append((mapping, fi0))
                continue
            
            a = a_digits[0] * 10 + a_digits[1]
            b = b_digits[0] * 10 + b_digits[1]
            
            # Try all formulas that match this equation
            matched = False
            for fi, (fam, name, fn, sp) in enumerate(FORMULAS):
                try:
                    val_str = fn(a, b)
                except: continue
                if val_str is None: continue
                
                if not matches(val_str, rhs, oc, sp): continue
                
                # Also check result symbols consistency with mapping
                if sp and is_sp:
                    r_digits = list(val_str)
                elif is_sp:
                    abs_str = str(abs(int(val_str))) if val_str.lstrip('-').isdigit() else val_str
                    r_digits = list(abs_str)
                elif is_sf:
                    r_digits = list(val_str)
                else:
                    r_digits = list(val_str)
                
                if len(r_digits) != len(rhs_body): continue
                
                ext_mapping = dict(mapping)
                r_ok = True
                for sym, d_ch in zip(rhs_body, r_digits):
                    d = int(d_ch)
                    if sym in op_chars: continue
                    if sym in ext_mapping:
                        if ext_mapping[sym] != d:
                            r_ok = False; break
                    else:
                        ext_mapping[sym] = d
                if not r_ok: continue
                
                vals = list(ext_mapping.values())
                if len(set(vals)) != len(vals): continue
                
                new_candidates.append((ext_mapping, fi0))
                matched = True
                break  # One match is enough to keep this candidate
            
            # If no formula matched, drop this candidate
        
        candidates = new_candidates
        if not candidates:
            return ''
    
    # Now try to compute query answer with surviving candidates
    for mapping, fi0 in candidates:
        # Check query symbols are mapped
        q_syms = [query[0], query[1], query[3], query[4]]
        if not all(s in mapping for s in q_syms if s not in op_chars):
            continue
        
        q_a = mapping.get(query[0], 0) * 10 + mapping.get(query[1], 0)
        q_b = mapping.get(query[3], 0) * 10 + mapping.get(query[4], 0)
        
        for fam, name, fn, sp in FORMULAS:
            try:
                val = fn(q_a, q_b)
                if matches(val, answer, q_op, sp):
                    return answer
            except: continue
    
    return ''


# ── Main Loop ────────────────────────────────────────────────────────────────

correct = 0; wrong = 0; unsolved = 0
results = []
t0 = time.time()

for i, p in enumerate(puzzles):
    q = p['query']
    is_numeric = q[:2].isdigit() and q[3:5].isdigit()
    
    if is_numeric:
        pred = solve_numeric(p)
    else:
        pred = solve_symbolic(p)
    
    if pred == p['answer']:
        correct += 1
    elif pred == '':
        unsolved += 1
    else:
        wrong += 1
    
    results.append({'id': p['id'], 'pred': pred, 'exp': p['answer'], 'ok': pred == p['answer']})
    
    if (i+1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"[{i+1}/{len(puzzles)}] ✓{correct} ✗{wrong} ?{unsolved} ({elapsed:.0f}s)")

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"FINAL: {correct}/{len(puzzles)} ({100*correct/len(puzzles):.1f}%) in {elapsed:.0f}s")
print(f"Wrong: {wrong}, Unsolved: {unsolved}")

wrong_samples = [r for r in results if not r['ok'] and r['pred'] != ''][:10]
if wrong_samples:
    print("\nSample wrong:")
    for r in wrong_samples:
        print(f"  {r['id']}: pred={r['pred']} exp={r['exp']}")

unsolved_samples = [r for r in results if r['pred'] == ''][:10]
if unsolved_samples:
    print(f"\nSample unsolved ({len([r for r in results if r['pred']==''])} total):")
    for r in unsolved_samples:
        print(f"  {r['id']}: exp={r['exp']}")